In [2]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np 
from torch.utils.data import Dataset
import random
import os

In [3]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [4]:
text="""In the beginning God created the heaven and the earth. And the earth was without form, and void; 
and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters. And God said, Let there be light:
and there was light. And God saw the light, that it was good: and God divided the light from the darkness. And God called the light Day, 
and the darkness he called Night. And the evening and the morning were the first day."""
text = text.lower()
print(f'Length of text: {len(text)} characters')
print(text[:100])

Length of text: 456 characters
in the beginning god created the heaven and the earth. and the earth was without form, and void; 
an


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for idx, ch in enumerate(chars)}
print(f'Unique characters: {chars}')
print(f'Vocabulary size: {vocab_size}')

Unique characters: ['\n', ' ', ',', '.', ':', ';', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']
Vocabulary size: 28


In [6]:
def encode(s):
    return [char_to_idx[ch] for ch in s]
def decode(indices):
    return ''.join([idx_to_char[idx] for idx in indices])
encoded_text = encode(text)
print(f'Encoded text: {encoded_text[:100]}')
print(f'Decoded text: {decode(encoded_text[:100])}')

Encoded text: [14, 18, 1, 23, 13, 10, 1, 7, 10, 12, 14, 18, 18, 14, 18, 12, 1, 12, 19, 9, 1, 8, 21, 10, 6, 23, 10, 9, 1, 23, 13, 10, 1, 13, 10, 6, 25, 10, 18, 1, 6, 18, 9, 1, 23, 13, 10, 1, 10, 6, 21, 23, 13, 3, 1, 6, 18, 9, 1, 23, 13, 10, 1, 10, 6, 21, 23, 13, 1, 26, 6, 22, 1, 26, 14, 23, 13, 19, 24, 23, 1, 11, 19, 21, 17, 2, 1, 6, 18, 9, 1, 25, 19, 14, 9, 5, 1, 0, 6, 18]
Decoded text: in the beginning god created the heaven and the earth. and the earth was without form, and void; 
an


In [7]:
seq_length = 50
input_sequences = []
target_sequences = []
for i in range(len(encoded_text) - seq_length):
    input_seq = encoded_text[i:i+seq_length]
    target_seq = encoded_text[i+1:i+seq_length+1]
    input_sequences.append(input_seq)
    target_sequences.append(target_seq)
print(f'Number of sequences: {len(input_sequences)}')
print(f'Example input sequence: {input_sequences[0]}')
print(f'Example target sequence: {target_sequences[0]}')
print(f'Example input text: {decode(input_sequences[0])}')
print(f'Example target text: {decode(target_sequences[0])}')

Number of sequences: 406
Example input sequence: [14, 18, 1, 23, 13, 10, 1, 7, 10, 12, 14, 18, 18, 14, 18, 12, 1, 12, 19, 9, 1, 8, 21, 10, 6, 23, 10, 9, 1, 23, 13, 10, 1, 13, 10, 6, 25, 10, 18, 1, 6, 18, 9, 1, 23, 13, 10, 1, 10, 6]
Example target sequence: [18, 1, 23, 13, 10, 1, 7, 10, 12, 14, 18, 18, 14, 18, 12, 1, 12, 19, 9, 1, 8, 21, 10, 6, 23, 10, 9, 1, 23, 13, 10, 1, 13, 10, 6, 25, 10, 18, 1, 6, 18, 9, 1, 23, 13, 10, 1, 10, 6, 21]
Example input text: in the beginning god created the heaven and the ea
Example target text: n the beginning god created the heaven and the ear
